# Exploratory Data Analysis of Customer Booking Data

This notebook contains a comprehensive exploratory data analysis of the British Airways customer booking dataset to understand patterns and relationships in the data before building predictive models.

## Objective
- Understand the structure and properties of the dataset
- Identify key patterns and relationships between variables
- Prepare the data for predictive modeling of booking completion

## 1. Data Loading and Initial Inspection

Let's start by loading the customer booking dataset and examining its structure.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Load the dataset
df = pd.read_csv("customer_booking.csv", encoding="ISO-8859-1")

# Display the first few rows
print("First 5 rows of the dataset:")
display(df.head())

In [ ]:
# Check the shape of the dataset
print(f"Dataset dimensions: {df.shape[0]} rows and {df.shape[1]} columns")

# Examine data types and check for missing values
print("\nDataset information:")
df.info()

In [ ]:
# Generate basic statistics for numerical columns
print("Basic statistics for numerical columns:")
display(df.describe())

# Check unique values for categorical columns
print("\nUnique values for categorical columns:")
for col in df.select_dtypes(include=['object']).columns:
    print(f"{col}: {df[col].nunique()} unique values")
    if df[col].nunique() < 15:  # Only show unique values if there aren't too many
        print(df[col].unique())
    print()

## 2. Data Cleaning and Preprocessing

Now, let's check for data quality issues like missing values, duplicate entries, and convert data types as needed.

In [ ]:
# Check for missing values
print("Missing values per column:")
display(df.isnull().sum())

# Check for duplicate rows
print(f"Number of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Convert flight_day from string to numeric
day_mapping = {
    "Mon": 1,
    "Tue": 2,
    "Wed": 3,
    "Thu": 4,
    "Fri": 5,
    "Sat": 6,
    "Sun": 7,
}

# Create a copy of the dataset with the mapped flight_day
df_clean = df.copy()
df_clean["flight_day_num"] = df_clean["flight_day"].map(day_mapping)

# Verify the conversion
print("Original flight_day values:")
print(df["flight_day"].value_counts())
print("\nConverted flight_day_num values:")
print(df_clean["flight_day_num"].value_counts())

# Keep the original column for reference

In [ ]:
# Check for outliers in numerical columns
numerical_cols = ['num_passengers', 'purchase_lead', 'length_of_stay', 
                  'flight_hour', 'flight_duration']

fig, axes = plt.subplots(len(numerical_cols), 1, figsize=(12, 15))
fig.tight_layout(pad=5.0)

for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df_clean[col], ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_xlabel(col)
    
plt.show()

## 3. Target Variable Analysis

Let's analyze the distribution of our target variable `booking_complete`.

In [ ]:
# Analyze the target variable distribution
target_counts = df_clean['booking_complete'].value_counts()
print("Target variable distribution:")
print(target_counts)
print(f"Proportion of completed bookings: {target_counts[1]/len(df_clean):.2%}")

plt.figure(figsize=(10, 6))
ax = sns.countplot(x='booking_complete', data=df_clean, palette='Blues')
plt.title('Distribution of Booking Completion', fontsize=16)
plt.xlabel('Booking Complete (0=No, 1=Yes)', fontsize=14)
plt.ylabel('Count', fontsize=14)

# Add count and percentage labels on top of the bars
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height} ({height/len(df_clean):.1%})',
            ha="center", fontsize=12)

plt.show()

## 4. Univariate Analysis of Features

Now let's explore the distributions of individual features, both numerical and categorical.

In [ ]:
# Univariate analysis for numerical features
plt.figure(figsize=(16, 20))

for i, col in enumerate(numerical_cols):
    plt.subplot(3, 2, i+1)
    sns.histplot(df_clean[col], kde=True)
    plt.title(f'Distribution of {col}', fontsize=14)
    plt.xlabel(col, fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    
plt.tight_layout()
plt.show()

In [ ]:
# Univariate analysis for categorical features
categorical_cols = ['sales_channel', 'trip_type', 'flight_day', 'route', 'booking_origin',
                   'wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals']

# For features with many unique values, we'll look at the top categories
for col in categorical_cols:
    plt.figure(figsize=(12, 6))
    
    if df_clean[col].nunique() > 15:
        # For columns with many unique values, show top 15
        top_categories = df_clean[col].value_counts().nlargest(15).index
        plot_data = df_clean[df_clean[col].isin(top_categories)]
        title_suffix = " (Top 15 Categories)"
    else:
        plot_data = df_clean
        title_suffix = ""
        
    ax = sns.countplot(y=col, data=plot_data, order=plot_data[col].value_counts().index)
    plt.title(f'Distribution of {col}{title_suffix}', fontsize=14)
    plt.xlabel('Count', fontsize=12)
    plt.ylabel(col, fontsize=12)
    
    # Add count labels
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 1, p.get_y() + p.get_height()/2, f'{width}', 
                ha='left', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

## 5. Bivariate Analysis with Target Variable

Let's examine relationships between each feature and the target variable `booking_complete`.

In [ ]:
# Bivariate analysis for numerical features vs. target
plt.figure(figsize=(16, 20))

for i, col in enumerate(numerical_cols):
    plt.subplot(3, 2, i+1)
    sns.boxplot(x='booking_complete', y=col, data=df_clean)
    plt.title(f'{col} by Booking Completion Status', fontsize=14)
    plt.xlabel('Booking Complete (0=No, 1=Yes)', fontsize=12)
    plt.ylabel(col, fontsize=12)
    
plt.tight_layout()
plt.show()

In [ ]:
# Creating more detailed visualizations for numerical features
for col in numerical_cols:
    plt.figure(figsize=(12, 6))
    
    # Create KDE plots for each class
    sns.kdeplot(data=df_clean[df_clean['booking_complete']==0][col], 
                label='Not Completed', shade=True, alpha=0.5)
    sns.kdeplot(data=df_clean[df_clean['booking_complete']==1][col], 
                label='Completed', shade=True, alpha=0.5)
    
    plt.title(f'Distribution of {col} by Booking Completion Status', fontsize=14)
    plt.xlabel(col, fontsize=12)
    plt.ylabel('Density', fontsize=12)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Bivariate analysis for categorical features vs. target
for col in categorical_cols:
    if df_clean[col].nunique() > 15:
        # For columns with many unique values, show top 10
        top_categories = df_clean[col].value_counts().nlargest(10).index
        plot_data = df_clean[df_clean[col].isin(top_categories)]
        title_suffix = " (Top 10 Categories)"
    else:
        plot_data = df_clean
        title_suffix = ""
    
    plt.figure(figsize=(14, 6))
    
    # Create a cross-tabulation of the categorical feature and target
    cross_tab = pd.crosstab(index=plot_data[col], columns=plot_data['booking_complete'])
    cross_tab_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0)
    
    # Plot the percentage of bookings completed for each category
    ax1 = plt.subplot(1, 2, 1)
    cross_tab_pct[1].sort_values().plot(kind='barh', color='skyblue')
    plt.title(f'Booking Completion Rate by {col}{title_suffix}', fontsize=14)
    plt.xlabel('Completion Rate', fontsize=12)
    plt.ylabel(col, fontsize=12)
    
    # Plot the count of each category by booking completion status
    ax2 = plt.subplot(1, 2, 2)
    sns.countplot(y=col, hue='booking_complete', data=plot_data, 
                  order=cross_tab.index, palette=['lightcoral', 'lightgreen'])
    plt.title(f'Count by {col} and Booking Status{title_suffix}', fontsize=14)
    plt.xlabel('Count', fontsize=12)
    plt.ylabel(col, fontsize=12)
    plt.legend(title='Booking Complete', labels=['No', 'Yes'])
    
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

Let's analyze correlations between numerical features and identify potential multicollinearity.

In [ ]:
# Create a correlation matrix for numerical features
corr_cols = numerical_cols + ['flight_day_num', 'booking_complete', 
                            'wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals']
corr_matrix = df_clean[corr_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Rank features by correlation with booking_complete
corr_with_target = corr_matrix['booking_complete'].drop('booking_complete').abs().sort_values(ascending=False)
print("Features ranked by absolute correlation with booking_complete:")
display(corr_with_target)

# Create a bar plot of correlations with target
plt.figure(figsize=(12, 8))
corr_with_target.plot(kind='bar', color='skyblue')
plt.title('Absolute Correlation of Features with Booking Completion', fontsize=16)
plt.xlabel('Feature', fontsize=14)
plt.ylabel('Absolute Correlation', fontsize=14)
plt.axhline(y=0.1, color='red', linestyle='--')  # Threshold line for reference
plt.tight_layout()
plt.show()

## 7. Categorical Features Analysis

Now let's do a more in-depth analysis of categorical variables and their impact on booking completion.

In [ ]:
# Calculate and plot booking completion rates for categorical features
important_cat_cols = ['sales_channel', 'trip_type', 'flight_day', 
                     'wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals']

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.tight_layout(pad=5.0)

for i, col in enumerate(important_cat_cols):
    row, col_idx = divmod(i, 2)
    
    # Calculate completion rate by category
    completion_rate = df_clean.groupby(col)['booking_complete'].mean().sort_values()
    counts = df_clean.groupby(col).size()
    
    # Plot completion rate
    ax = completion_rate.plot(kind='barh', ax=axes[row, col_idx], color='skyblue')
    axes[row, col_idx].set_title(f'Booking Completion Rate by {col}', fontsize=14)
    axes[row, col_idx].set_xlabel('Completion Rate', fontsize=12)
    axes[row, col_idx].set_ylabel(col, fontsize=12)
    
    # Add count and percentage annotations
    for j, p in enumerate(ax.patches):
        category = completion_rate.index[j]
        count = counts[category]
        percentage = p.get_width() * 100
        ax.text(p.get_width() + 0.01, p.get_y() + p.get_height()/2, 
                f'  {percentage:.1f}% (n={count})', va='center', fontsize=10)
    
plt.show()

In [ ]:
# Analyze route and booking_origin features (which may have many categories)
for col in ['route', 'booking_origin']:
    # Get top 15 categories by frequency
    top_categories = df_clean[col].value_counts().nlargest(15).index
    plot_data = df_clean[df_clean[col].isin(top_categories)]
    
    # Calculate completion rate by category
    completion_rate = plot_data.groupby(col)['booking_complete'].mean().sort_values()
    counts = plot_data.groupby(col).size()
    
    plt.figure(figsize=(14, 10))
    ax = completion_rate.plot(kind='barh', color='skyblue')
    plt.title(f'Booking Completion Rate by {col} (Top 15 Categories)', fontsize=14)
    plt.xlabel('Completion Rate', fontsize=12)
    plt.ylabel(col, fontsize=12)
    
    # Add count and percentage annotations
    for i, p in enumerate(ax.patches):
        category = completion_rate.index[i]
        count = counts[category]
        percentage = p.get_width() * 100
        ax.text(p.get_width() + 0.01, p.get_y() + p.get_height()/2, 
                f'  {percentage:.1f}% (n={count})', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()

## 8. Time-Based Patterns Analysis

Finally, let's investigate temporal patterns in the data using features like flight_hour, flight_day, purchase_lead, and length_of_stay.

In [ ]:
# Analyze booking completion by flight hour
plt.figure(figsize=(14, 6))
hour_completion = df_clean.groupby('flight_hour')['booking_complete'].mean()
hour_counts = df_clean.groupby('flight_hour').size()

ax = plt.subplot(1, 1, 1)
ax2 = ax.twinx()

# Plot completion rate line
ax.plot(hour_completion.index, hour_completion.values, 'o-', color='blue', linewidth=2, label='Completion Rate')
ax.set_xlabel('Flight Hour', fontsize=12)
ax.set_ylabel('Booking Completion Rate', color='blue', fontsize=12)
ax.set_ylim([0, max(hour_completion.values) * 1.2])
ax.tick_params(axis='y', labelcolor='blue')

# Plot count bars
ax2.bar(hour_counts.index, hour_counts.values, alpha=0.3, color='gray', label='Number of Bookings')
ax2.set_ylabel('Number of Bookings', color='gray', fontsize=12)
ax2.tick_params(axis='y', labelcolor='gray')

# Add title and legend
plt.title('Booking Completion Rate by Flight Hour', fontsize=16)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze booking completion by flight day
plt.figure(figsize=(14, 6))
day_completion = df_clean.groupby('flight_day')['booking_complete'].mean()
day_counts = df_clean.groupby('flight_day').size()

ax = plt.subplot(1, 1, 1)
ax2 = ax.twinx()

# Plot completion rate line
ax.plot(day_completion.index, day_completion.values, 'o-', color='green', linewidth=2, label='Completion Rate')
ax.set_xlabel('Flight Day', fontsize=12)
ax.set_ylabel('Booking Completion Rate', color='green', fontsize=12)
ax.set_ylim([0, max(day_completion.values) * 1.2])
ax.tick_params(axis='y', labelcolor='green')

# Plot count bars
ax2.bar(day_counts.index, day_counts.values, alpha=0.3, color='gray', label='Number of Bookings')
ax2.set_ylabel('Number of Bookings', color='gray', fontsize=12)
ax2.tick_params(axis='y', labelcolor='gray')

# Add title and legend
plt.title('Booking Completion Rate by Flight Day', fontsize=16)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze booking completion by purchase lead time (grouped in bins)
plt.figure(figsize=(16, 6))

# Create bins for purchase lead
bins = [0, 7, 30, 60, 90, 180, 365, df_clean['purchase_lead'].max()]
labels = ['0-7 days', '8-30 days', '31-60 days', '61-90 days', '91-180 days', '181-365 days', '365+ days']
df_clean['lead_time_group'] = pd.cut(df_clean['purchase_lead'], bins=bins, labels=labels)

lead_completion = df_clean.groupby('lead_time_group')['booking_complete'].mean()
lead_counts = df_clean.groupby('lead_time_group').size()

ax = plt.subplot(1, 1, 1)
ax2 = ax.twinx()

# Plot completion rate line
ax.plot(lead_completion.index, lead_completion.values, 'o-', color='purple', linewidth=2, label='Completion Rate')
ax.set_xlabel('Purchase Lead Time', fontsize=12)
ax.set_ylabel('Booking Completion Rate', color='purple', fontsize=12)
ax.set_ylim([0, max(lead_completion.values) * 1.2])
ax.tick_params(axis='y', labelcolor='purple')

# Plot count bars
ax2.bar(range(len(lead_counts)), lead_counts.values, alpha=0.3, color='gray', label='Number of Bookings')
ax2.set_xticks(range(len(lead_counts)))
ax2.set_xticklabels(lead_counts.index, rotation=45)
ax2.set_ylabel('Number of Bookings', color='gray', fontsize=12)
ax2.tick_params(axis='y', labelcolor='gray')

# Add title and legend
plt.title('Booking Completion Rate by Purchase Lead Time', fontsize=16)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze booking completion by length of stay (grouped in bins)
plt.figure(figsize=(16, 6))

# Create bins for length of stay
bins = [-1, 0, 1, 3, 7, 14, 30, df_clean['length_of_stay'].max()]
labels = ['0 days', '1 day', '2-3 days', '4-7 days', '8-14 days', '15-30 days', '31+ days']
df_clean['stay_group'] = pd.cut(df_clean['length_of_stay'], bins=bins, labels=labels)

stay_completion = df_clean.groupby('stay_group')['booking_complete'].mean()
stay_counts = df_clean.groupby('stay_group').size()

ax = plt.subplot(1, 1, 1)
ax2 = ax.twinx()

# Plot completion rate line
ax.plot(stay_completion.index, stay_completion.values, 'o-', color='orange', linewidth=2, label='Completion Rate')
ax.set_xlabel('Length of Stay', fontsize=12)
ax.set_ylabel('Booking Completion Rate', color='orange', fontsize=12)
ax.set_ylim([0, max(stay_completion.values) * 1.2])
ax.tick_params(axis='y', labelcolor='orange')

# Plot count bars
ax2.bar(range(len(stay_counts)), stay_counts.values, alpha=0.3, color='gray', label='Number of Bookings')
ax2.set_xticks(range(len(stay_counts)))
ax2.set_xticklabels(stay_counts.index, rotation=45)
ax2.set_ylabel('Number of Bookings', color='gray', fontsize=12)
ax2.tick_params(axis='y', labelcolor='gray')

# Add title and legend
plt.title('Booking Completion Rate by Length of Stay', fontsize=16)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

## Summary of Key Findings

Based on the exploratory data analysis, we can draw the following key insights:

1. **Target Distribution**: The dataset shows an imbalance in booking completion, with approximately X% of bookings being completed.

2. **Influential Features**: The features most strongly correlated with booking completion are:
   - [List top features based on correlation analysis]

3. **Temporal Patterns**: 
   - [Summarize findings about flight hour, day, purchase lead time]
   - [Mention any weekly patterns discovered]

4. **Customer Preferences**:
   - [Summarize findings about extra baggage, preferred seating, in-flight meals]
   - [Mention how these relate to booking completion]

5. **Trip Characteristics**:
   - [Summarize findings about length of stay, trip type, etc.]
   - [Mention how these relate to booking completion]

These insights will guide our feature engineering and modeling approach for predicting booking completion.